# S2 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa
## Proyecto Sello — Contaminación del Agua

Actividad individual (equivalente a la sección 4.1 de la guía S2), aplicada a un único archivo `lecturas_agua.csv` con **1,500,000 lecturas** de sensores de agua en 4 puntos: Río Coata, Juliaca (urbano), Cabanillas (ciudad) y Cabanillas (nacimiento de agua).

Cubre los 5 parámetros de calidad de agua del proyecto: **pH**, **turbidez**, **metales pesados** (plomo, arsénico, mercurio, cadmio), **bacterias y parásitos** (coliformes fecales, *E. coli*, presencia de parásitos) y **radiactividad**.

## 1. SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("proyecto-agua-fundamentos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

In [ ]:
ORIGEN_DATOS = "/opt/s02-fundamentos/data"  # ajusta segun donde tengas el CSV

## 2. Cargar y explorar (primera lectura: `inferSchema`)

`sensor_id` es una cadena de solo dígitos (`"0001"`, `"0002"`, ...) — con `inferSchema=True` Spark puede leerla como número y perder el cero inicial (el mismo riesgo que `article_id` en la guía del curso).

In [ ]:
df_agua = spark.read.csv(
    f"{ORIGEN_DATOS}/lecturas_agua.csv",
    header=True,
    inferSchema=True,
)

df_agua.printSchema()
df_agua.show(5, truncate=False)

Verifica si `sensor_id` perdió el cero inicial. Corrígelo con un esquema explícito:

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

schema_agua = StructType([
    StructField("id_lectura", IntegerType(), True),
    StructField("sensor_id", StringType(), True),        # <- texto, conserva el cero inicial
    StructField("ubicacion", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
    StructField("canal_transmision_id", IntegerType(), True),
    StructField("ph", DoubleType(), True),
    StructField("turbidez_ntu", DoubleType(), True),
    StructField("temperatura_c", DoubleType(), True),
    StructField("conductividad_us_cm", DoubleType(), True),
    StructField("solidos_disueltos_totales_mg_l", DoubleType(), True),
    StructField("oxigeno_disuelto_mg_l", DoubleType(), True),
    StructField("plomo_mg_l", DoubleType(), True),
    StructField("arsenico_mg_l", DoubleType(), True),
    StructField("mercurio_mg_l", DoubleType(), True),
    StructField("cadmio_mg_l", DoubleType(), True),
    StructField("coliformes_fecales_nmp_100ml", DoubleType(), True),
    StructField("escherichia_coli_nmp_100ml", DoubleType(), True),
    StructField("presencia_parasitos", IntegerType(), True),
    StructField("radiactividad_bq_l", DoubleType(), True),
    StructField("caudal_l_s", DoubleType(), True),
    StructField("indice_riesgo_normalizado", DoubleType(), True),
])

df_agua = spark.read.csv(
    f"{ORIGEN_DATOS}/lecturas_agua.csv",
    header=True,
    schema=schema_agua,
)

df_agua.printSchema()

In [ ]:
num_filas, num_cols = df_agua.count(), len(df_agua.columns)
print(f"Filas: {num_filas}, Columnas: {num_cols}")

df_agua.select(
    "ph", "turbidez_ntu", "plomo_mg_l", "radiactividad_bq_l", "escherichia_coli_nmp_100ml"
).describe().show()

## 3. Transformaciones y evaluación perezosa

`select()` y `filter()` encadenados no ejecutan nada todavía — solo construyen el plan.

In [ ]:
from pyspark.sql.functions import col

df_transformado = (
    df_agua
    .select("ubicacion", "sensor_id", "fecha_hora", "ph", "plomo_mg_l", "radiactividad_bq_l")
    .filter(col("ubicacion") == "Rio Coata")
)

# Hasta aqui NO se ejecuto nada: solo se construyo el plan
df_transformado

In [ ]:
# Accion: aqui recien Spark ejecuta
df_transformado.show(10, truncate=False)
df_transformado.count()

## 4. Plan de ejecución con `explain()`

In [ ]:
df_transformado.explain(True)

Compara el **Parsed Logical Plan** contra el **Optimized Logical Plan**: si Catalyst aplicó predicate pushdown, el `Filter` va a aparecer antes del `Project`, aunque en tu código el `select()` estuviera escrito primero. Revisa también si aparece `PushedFilters` en el Physical Plan.

## 5. Funciones con `withColumn()` — clasificar por los 5 parámetros

Umbrales de referencia: pH aceptable 6.5-8.5; plomo > 0.01 mg/L y arsénico > 0.01 mg/L se consideran altos (referencia OMS); presencia de parásitos es cualquier detección positiva; radiactividad > 0.5 Bq/L se marca como alta en este proyecto.

In [ ]:
from pyspark.sql.functions import col, when, lit, current_date

df_agua = df_agua.withColumn(
    "ph_dentro_rango_potable",
    when((col("ph") >= 6.5) & (col("ph") <= 8.5), "Si").otherwise("No")
)

df_agua = df_agua.withColumn(
    "riesgo_metales",
    when((col("plomo_mg_l") > 0.01) | (col("arsenico_mg_l") > 0.01), "Alto")
    .when((col("plomo_mg_l") > 0.005) | (col("arsenico_mg_l") > 0.005), "Moderado")
    .otherwise("Bajo")
)

df_agua = df_agua.withColumn(
    "riesgo_biologico",
    when((col("presencia_parasitos") == 1) | (col("escherichia_coli_nmp_100ml") > 500), "Alto")
    .when(col("coliformes_fecales_nmp_100ml") > 500, "Moderado")
    .otherwise("Bajo")
)

df_agua = df_agua.withColumn(
    "nivel_radiactividad",
    when(col("radiactividad_bq_l") > 0.5, "Alto")
    .when(col("radiactividad_bq_l") > 0.2, "Moderado")
    .otherwise("Bajo")
)

df_agua = df_agua.withColumn("fuente_dato", lit("Sensor IoT - Proyecto Sello"))
df_agua = df_agua.withColumn("fecha_procesado", current_date())

df_agua.select(
    "ubicacion", "ph", "ph_dentro_rango_potable", "riesgo_metales",
    "riesgo_biologico", "nivel_radiactividad"
).show(10, truncate=False)

## 6. Agrupaciones y agregaciones

In [ ]:
from pyspark.sql.functions import avg, count, max as spark_max

resumen_por_ubicacion = df_agua.groupBy("ubicacion").agg(
    count("*").alias("num_lecturas"),
    avg("ph").alias("ph_promedio"),
    avg("plomo_mg_l").alias("plomo_promedio"),
    spark_max("plomo_mg_l").alias("plomo_maximo"),
    avg("escherichia_coli_nmp_100ml").alias("ecoli_promedio"),
    avg("presencia_parasitos").alias("proporcion_con_parasitos"),
    avg("radiactividad_bq_l").alias("radiactividad_promedio"),
)

resumen_por_ubicacion.orderBy(col("radiactividad_promedio").desc()).show(truncate=False)

**Advertencia de dominio:** `indice_riesgo_normalizado` es un índice compuesto normalizado a `[0, 1]` — no es una concentración real ni un estándar regulatorio. `presencia_parasitos` tampoco se debe promediar como si fuera continua sin aclarar que el promedio representa una *proporción* de lecturas positivas (0 a 1), no una concentración.

Función ventana — a diferencia de `groupBy().agg()`, no colapsa filas; cada lectura conserva su fila y trae el promedio de su ubicación:

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg as avg_

ventana_ubicacion = Window.partitionBy("ubicacion")

df_con_promedio = df_agua.withColumn(
    "ph_promedio_ubicacion",
    avg_("ph").over(ventana_ubicacion)
)

df_con_promedio.select("ubicacion", "sensor_id", "ph", "ph_promedio_ubicacion").show(10, truncate=False)

## 7. RDD — `map`, `flatMap`, `filter`, `reduceByKey`

Por cada lectura, identifica qué parámetros salieron fuera de rango (puede ser ninguno, uno o varios) y cuenta, sobre las 1,500,000 lecturas, cuántas veces se excedió cada parámetro en total — un solo `reduceByKey` sobre todo el dataset.

In [ ]:
def problemas_de_la_lectura(fila):
    problemas = []
    if fila.ph < 6.5 or fila.ph > 8.5:
        problemas.append("ph_fuera_de_rango")
    if fila.plomo_mg_l > 0.01:
        problemas.append("plomo_alto")
    if fila.arsenico_mg_l > 0.01:
        problemas.append("arsenico_alto")
    if fila.mercurio_mg_l > 0.001:
        problemas.append("mercurio_alto")
    if fila.presencia_parasitos == 1:
        problemas.append("parasitos_presentes")
    if fila.radiactividad_bq_l > 0.5:
        problemas.append("radiactividad_alta")
    return problemas

rdd_lecturas = df_agua.select(
    "ph", "plomo_mg_l", "arsenico_mg_l", "mercurio_mg_l", "presencia_parasitos", "radiactividad_bq_l"
).rdd

# flatMap: cada lectura puede producir 0, 1 o varios "problemas" -> se aplanan en un solo RDD
rdd_problemas = rdd_lecturas.flatMap(problemas_de_la_lectura)

# filter: descarta cualquier valor vacio (no deberia haber, pero es la practica pedida)
rdd_problemas = rdd_problemas.filter(lambda p: p != "")

# map: convierte cada problema en un par (problema, 1)
pares = rdd_problemas.map(lambda p: (p, 1))

# reduceByKey: suma cuantas veces aparecio cada tipo de problema en todo el dataset
from operator import add
conteo_problemas = pares.reduceByKey(add)

conteo_problemas.takeOrdered(10, key=lambda x: -x[1])

**Reto adicional:** cuenta, por ubicación, cuántas lecturas tienen `riesgo_biologico == "Alto"`, usando el mismo patrón `map`/`filter`/`reduceByKey`:

In [ ]:
rdd_riesgo = df_agua.select("ubicacion", "riesgo_biologico").rdd

pares_riesgo_alto = (
    rdd_riesgo
    .filter(lambda fila: fila.riesgo_biologico == "Alto")
    .map(lambda fila: (fila.ubicacion, 1))
)

conteo_riesgo_alto = pares_riesgo_alto.reduceByKey(add)
conteo_riesgo_alto.collect()

## 8. Documentar hallazgos y reflexión

Agrega celdas markdown debajo de cada bloque explicando qué hiciste y qué observaste — es la base de tu informe.

**Reflexión técnica breve (5-8 líneas):** ¿por qué la evaluación perezosa es útil para procesar 1.5 millones de lecturas de sensores de agua, y qué pasaría si Spark ejecutara cada `withColumn()`/`filter()` de inmediato, apenas se escribe?